In [40]:
import pandas as pd
from collections import defaultdict
import random
from tqdm import tqdm
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score as span_f1_score
from sklearn.metrics import f1_score

In [41]:
LabelToID = {
    'O': 0,
    'B-Date': 1,
    'I-Date': 2,
    'B-Person': 3,
    'I-Person': 4,
    'B-Location': 5,
    'I-Location': 6,
    'B-Facility': 7,
    'I-Facility': 8,
    'B-Organization': 9,
    'I-Organization': 10,
    'B-Misc': 11,
    'I-Misc': 12,
    'B-Money': 13,
    'I-Money': 14,
    'B-NORP': 15,
    'I-NORP': 16,
    'B-Product': 17,
    'I-Product': 18}

In [42]:
train_eng = pd.read_parquet('trainEng.parquet')
test_eng = pd.read_parquet('testEng.parquet')
train_ww = pd.read_parquet('trainGlobal.parquet')
test_ww=pd.read_parquet('testGlobal.parquet')
train_combined=pd.read_parquet('trainCombined.parquet')

In [43]:
## ----- MODEL -----
# Make vocab
def build_vocab(datasets_list): # Vocabulary mapping words to integer IDs
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for df in datasets_list:
        for tokens in df['tokens']:
            for word in tokens:
                if word not in word2idx:
                    word2idx[word] = len(word2idx)
    return word2idx

# Dataset class
class NERDataset(Dataset):      # PyTorch Dataset Class
    def __init__(self, df, word2idx):
        self.sentences = df['tokens'].tolist()
        self.labels = df['ner_tags'].tolist()
        self.word2idx = word2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):     # Convert words to indices, fallback to <UNK> if word is unseen
        word_indices = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in self.sentences[idx]]
        label_indices = self.labels[idx]
        return torch.tensor(word_indices), torch.tensor(label_indices)

def collate_fn(batch):      # function to pad sentences to the same length in a batch
    sentences, labels = zip(*batch)
    padded_sentences = pad_sequence(sentences, batch_first=True, padding_value=0)   # Pad sentences with 0 (<PAD>)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)      # Pad labels with -100 (PyTorch CrossEntropyLoss ignores -100 by default)
    return padded_sentences, padded_labels

# Bi-LSTM model
class RNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(RNN_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)     # Word embedding layer
        
        self.lstm = nn.LSTM(            # Bidirectional LSTM
            embedding_dim, 
            hidden_dim, 
            batch_first=True, 
            bidirectional=True)
        
        # Linear layer mapping hidden states to tag classes
        self.fc = nn.Linear(hidden_dim * 2, num_classes)        # hidden_dim * 2 because it's bidirectional

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(lstm_out)
        return logits
    
# Model training    
def train_model(model, dataloader, optimizer, criterion, device, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for sentences, labels in dataloader:
            sentences, labels = sentences.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = model(sentences)
            
            # Reshape logits and labels for CrossEntropyLoss
            logits = logits.view(-1, logits.shape[-1])      # logits: (batch_size * seq_len, num_classes)
            labels = labels.view(-1)        # labels: (batch_size * seq_len)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

# Model evaluation
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    sentence_preds = []
    
    with torch.no_grad():
        for sentences, labels in dataloader:
            sentences, labels = sentences.to(device), labels.to(device)
            logits = model(sentences)
            
            preds = torch.argmax(logits, dim=-1)        # Get the predicted class indices
            
            for pred_row, label_row in zip(preds, labels):
                mask = label_row != -100
                filtered = pred_row[mask].cpu().numpy()
                sentence_preds.append(filtered.tolist())  # one list per sentence
                all_preds.extend(filtered)
                all_labels.extend(label_row[mask].cpu().numpy())

            # preds = preds.view(-1).cpu().numpy()        # Flatten to compare
            # labels = labels.view(-1).cpu().numpy()
            
            # mask = labels != -100           # Filter out the padded tokens (-100)
            # all_preds.extend(preds[mask])
            # all_labels.extend(labels[mask])
            
    f1 = f1_score(all_labels, all_preds, average='macro')
    return f1,all_preds,sentence_preds

# Create a reverse dictionary to map integer predictions back to string labels
id2label = {v: k for k, v in LabelToID.items()}

word2idx = build_vocab([train_eng, train_ww])
vocab_size = len(word2idx)
num_classes = len(LabelToID)

In [44]:
def build_tag_index(ner_tags_series):
    """Build {tag: [row_indices]} index once, instead of scanning every iteration."""
    index = defaultdict(list)
    for i, tags in enumerate(ner_tags_series):
        for tag in set(tags):  # set() avoids duplicate row entries per tag
            if tag != 0:
                index[tag].append(i)
    return index

def switchTokensNew(df1: pd.DataFrame, df2: pd.DataFrame, n: int):
    df1 = df1.copy()
    df2 = df2.copy()

    # Build indexes
    idx1 = build_tag_index(df1['ner_tags'])
    idx2 = build_tag_index(df2['ner_tags'])

    all_tags = list(set(idx1.keys()) & set(idx2.keys()))  # Only tags present in BOTH
    if not all_tags:
        return df1, df2

    # Convert token columns to lists once for faster .at access
    tokens1_col = df1['tokens'].tolist()
    tokens2_col = df2['tokens'].tolist()
    tags1_col   = df1['ner_tags'].tolist()
    tags2_col   = df2['ner_tags'].tolist()

    for _ in tqdm(range(n)):
        tag = random.choice(all_tags)

        s1 = random.choice(idx1[tag])
        s2 = random.choice(idx2[tag])

        tokens1 = list(tokens1_col[s1])
        tokens2 = list(tokens2_col[s2])
        tags1   = tags1_col[s1]
        tags2   = tags2_col[s2]

        i1 = random.choice([i for i, t in enumerate(tags1) if t == tag])
        i2 = random.choice([i for i, t in enumerate(tags2) if t == tag])

        tokens1[i1], tokens2[i2] = tokens2[i2], tokens1[i1]

        tokens1_col[s1] = tokens1
        tokens2_col[s2] = tokens2

    df1['tokens'] = tokens1_col
    df2['tokens'] = tokens2_col
    return df1, df2

train_eng_switched,train_ww_switched=switchTokensNew(train_eng,train_ww,100000) #TODO: Graph with n switches
train_combined_switched = pd.concat([train_eng_switched, train_ww_switched], ignore_index=True)

100%|██████████| 100000/100000 [00:01<00:00, 84541.60it/s]


In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create DataLoaders
train_ds = NERDataset(train_eng, word2idx)
test_ds = NERDataset(test_ww, word2idx)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
# Initialize a fresh model for each experiment
model = RNN_NER(vocab_size, embedding_dim=128, hidden_dim=256, num_classes=num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-100) # Ignores padding!
    
# Train
train_model(model, train_dl, optimizer, criterion, device, epochs=5)
    
# Evaluate
f1_score_val,_,predictions = evaluate_model(model, test_dl, device)
print(f"Result -> F1 Score: {f1_score_val:.4f}")

Epoch 1/5 | Loss: 0.3256
Epoch 2/5 | Loss: 0.1427
Epoch 3/5 | Loss: 0.0867
Epoch 4/5 | Loss: 0.0513
Epoch 5/5 | Loss: 0.0294
Result -> F1 Score: 0.3215


In [46]:
test_ww['region']=test_ww['region']
test_ww['predicted_tags'] = [
    [i for i in sent] for sent in predictions
]

In [47]:
test_ww

,tokens,ner_tags,region,predicted_tags
12496,"[He, added, ,, “, Iran, 's, strategy, in, buil...","[0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",iran,"[0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
20345,"[Family, members, of, the, Kang, Jian, founder...","[0, 0, 0, 0, 9, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",tw,"[0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
19193,"[Cape, Town, -, Following, President, Cyril, R...","[5, 6, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, ...",sa,"[3, 4, 0, 4, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2123,"[For, his, part, ,, the, Moudjahid, Mohamed, T...","[0, 0, 0, 0, 0, 3, 4, 4, 4, 0, 0, 0, 0, 0, 0, ...",alg,"[0, 0, 0, 0, 0, 0, 10, 0, 9, 0, 0, 0, 0, 0, 0,..."
21069,"[Protesting, Under, a, Dictatorship]","[0, 0, 0, 0]",ven,"[0, 0, 0, 0]"
...,...,...,...,...
10125,"[The, story, of, Imam, Ahmad, ’s, campaigns, w...","[0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0,...",et,"[0, 0, 0, 3, 4, 11, 0, 0, 0, 0, 0, 0, 11, 0, 0..."
2749,"[The, trek, to, the, cave, traverses, forests,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",asia,"[0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7453,"[The, department, also, said, there, was, no, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, ...",cn,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
17247,"["", You, fall, to, Moana, Jackson, 's, situati...","[0, 0, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nz,"[0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
